<a href="https://colab.research.google.com/github/peijinUQAM/wildfire-gpd-pipeline/blob/main/gpd_ros_pipeline_v6_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GBM-GPD Extreme Fire Spread Modeling — v6 (Fixed)

This notebook corrects all bugs and mathematical inconsistencies identified in v5.
Changes from v5 are marked with `# FIX:` comments.

**Summary of fixes**

| # | Location | Issue | Fix |
|---|----------|-------|-----|
| 1 | `return_level` | Missing threshold `u` — all RL values underestimated | Added `u` parameter |
| 2 | `run_return_level_analysis` | T labeled "yr" but treated as fire-days; wrong T values | Renamed to `_fd`, T ∈ {10,50,100,250} fire-days |
| 3 | `compute_main_results` | State dict missing keys → downstream KeyErrors | All keys restored |
| 4 | Split design | Triple-dipping: val set used for early stopping, ξ selection, and evaluation | Three-way split: train / tune / eval |
| 5 | `_fit_single_fold` | Spatial CV used test fold for early stopping | Temporal tune subset used instead |
| 6 | `build_fold_labels` | `min_size` parameter ignored; small ecozones not merged | `min_size` enforced |
| 7 | Paper vs code | Paper claimed EBM on full data; code correctly used train-only | Comment corrected |
| 8 | `_fit_single_fold` | Spatial CV used global ξ, not fold-specific | `SPATIAL_CV_REFIT_XI` flag; fold-level ξ when enabled |
| 9 | Cell 30 | `compute_main_results` defined twice | Dead copy removed |
| 10 | `GPDDist` | Missing `logpdf` method → AttributeError | `logpdf` added |
| 11 | `run_shap_analysis` | f_interp alongside its own inputs → SHAP collinearity | SHAP run on non-augmented matrix |


## 1. Environment

In [ ]:
!pip install -q lightgbm interpret shap google-cloud-bigquery-storage pyarrow scipy cartopy ngboost geopandas fiona pyogrio shapely

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 65.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import shap

from scipy.optimize import minimize_scalar
from scipy.stats import genpareto
from sklearn.metrics import mean_pinball_loss

warnings.filterwarnings("ignore")
np.random.seed(42)

print(f"LightGBM version: {lgb.__version__}")


## 2. Configuration

In [ ]:
PROJECT_ID = "wildfireearth"
DATASET    = "wildfire_analysis"
TABLE      = "firegrowth_groups_v3"

RESPONSE_COL = "sprdistm"
GROUP_COL    = "ecozone"
YEAR_COL     = "year"
DOY_COL      = "DOB"

TAU              = 0.90
EXCEEDANCE_FLOOR = 500.0
CAP_PERCENTILE   = 99.0
XI_GRID          = np.linspace(0.05, 0.90, 18)

# FIX 4: Three-way temporal split.
#   TRAIN  — model fitting
#   TUNE   — early stopping + xi selection (never reported as a test metric)
#   EVAL   — final held-out evaluation only (touches no fitting step)
TUNE_YEARS = [2018, 2019]
EVAL_YEARS = [2020, 2021]
# Everything else is TRAIN (2002–2017).

# FIX 8: Set True for publication — re-estimates xi inside each spatial CV fold.
# Slow (18 LightGBM runs per fold) but removes leakage of global xi into CV.
SPATIAL_CV_REFIT_XI = False

ECOZONE_NAMES = {
    1: "Arctic Cordillera",  2: "Northern Arctic",   3: "Southern Arctic",
    4: "Taiga Plains",       5: "Taiga Shield W",    6: "Boreal Shield W",
    7: "Atlantic Maritime",  8: "Mixedwood Plains",  9: "Boreal Plains",
    10: "Prairies",          11: "Taiga Cordillera", 12: "Boreal Cordillera",
    13: "Pacific Maritime",  14: "Montane Cordillera", 15: "Hudson Plains",
    25: "Taiga Shield E",    26: "Boreal Shield E",
    999: "North (merged)",
}

OUTPUT_DIR             = Path("output/gbm_gpd_v6")
RUN_SPATIAL_CV         = True
RUN_BIOME_CV           = True
RUN_NGBOOST_COMPARISON = True
RUN_SHAP_ANALYSIS      = True
RUN_RETURN_LEVELS      = True
RUN_MEAN_EXCESS        = True
FAST_CV                = True        # 500 EBM rounds per fold; set False for publication
RANDOM_STATE           = 42

RUN_CHOROPLETH_MAPS    = True
ECOZONE_SHAPEFILE_PATH = None        # e.g. "data/canada_ecozones.shp"
ECOZONE_ID_COLUMN      = "ecozone"
BIOME_NAME_COLUMN      = None


## 3. Data loading

In [ ]:
BQ_QUERY = f"""
SELECT
  ID, year, DOB, ecozone, lat, lon, sprdistm,
  fwi, isi, ffmc, dmc, dc, bui, fwi_prev1, fwi_prev2, d_fwi, d_isi,
  ws, rh, tmax, vpd, prec, rh_prev1, rh_prev2, d_ws, d_rh, d_vpd, d_tmax, d_prec,
  Biomass, Closure, prcC, prcB, peattype, peat10,
  nonfuel1k, nonfuel2k, nonfuel5k, nonfuel10k,
  roaddens2k, roaddens10k, roaddens25k, roaddist,
  hydrodens2k, hydrodens5k, hydrodens10k, hydrodens25k,
  dem, slope, aspect_cos, aspect_sin, twi
FROM `{PROJECT_ID}.{DATASET}.{TABLE}`
WHERE sprdistm > 0
  AND sprdistm IS NOT NULL
"""


def load_from_bigquery(project_id: str = PROJECT_ID,
                       query: str = BQ_QUERY,
                       authenticate: bool = True) -> pd.DataFrame:
    if authenticate:
        try:
            from google.colab import auth
            auth.authenticate_user()
        except Exception:
            pass
    from google.cloud import bigquery
    client = bigquery.Client(project=project_id)
    return client.query(query).to_dataframe()


## 4. Covariates and preprocessing

In [ ]:
INTERPRETED = ["isi", "fwi", "ws", "rh", "slope", "doy_sin", "doy_cos"]

NON_INTERPRETED = [
    "ffmc", "dmc", "dc", "bui",
    "fwi_prev1", "fwi_prev2", "rh_prev1", "rh_prev2",
    "d_fwi", "d_isi", "d_ws", "d_rh", "d_vpd", "d_tmax", "d_prec",
    "tmax", "vpd", "prec",
    "Biomass", "Closure", "prcC", "prcB", "peattype", "peat10",
    "nonfuel1k", "nonfuel2k", "nonfuel5k", "nonfuel10k",
    "roaddens2k", "roaddens10k", "roaddens25k", "roaddist",
    "hydrodens2k", "hydrodens5k", "hydrodens10k", "hydrodens25k",
    "dem", "twi", "aspect_cos", "aspect_sin", "lat", "lon",
]

ALL_FEATURES = INTERPRETED + NON_INTERPRETED


def preprocess_fire_days(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    doy = out[DOY_COL].astype(float).fillna(1.0)
    out["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    out["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    if GROUP_COL in out.columns:
        out[GROUP_COL] = out[GROUP_COL].astype("category")
    if "peattype" in out.columns:
        out["peattype"] = out["peattype"].astype("category")
    usable = [c for c in ALL_FEATURES if c in out.columns]
    mask = out[usable].notna().all(axis=1) & out[RESPONSE_COL].notna()
    if GROUP_COL in out.columns:
        mask &= out[GROUP_COL].notna()
    return out.loc[mask].reset_index(drop=True)


def summarize_dataset(df: pd.DataFrame) -> None:
    y = df[RESPONSE_COL].to_numpy(dtype=float)
    print(f"Clean dataset  : {len(df):,} fire-days")
    print(f"Median spread  : {np.median(y):.0f} m/day")
    print(f"95th pct spread: {np.percentile(y, 95):.0f} m/day")
    print(f"Max spread     : {np.max(y):.0f} m/day")
    n_train = (~df[YEAR_COL].isin(TUNE_YEARS + EVAL_YEARS)).sum()
    n_tune  = df[YEAR_COL].isin(TUNE_YEARS).sum()
    n_eval  = df[YEAR_COL].isin(EVAL_YEARS).sum()
    print(f"\nSplit sizes — train: {n_train:,}  tune: {n_tune:,}  eval: {n_eval:,}")


## 5. Validation design

### Three-way temporal split (Fix 4)

| Set | Years | Role |
|-----|-------|------|
| **Train** | 2002–2017 | Model fitting |
| **Tune** | 2018–2019 | Early stopping + ξ selection via profile likelihood |
| **Eval** | 2020–2021 | Final held-out evaluation — never touches any fitting step |

Previously the tune and eval roles were collapsed into a single 2020–2021 set,
creating triple-dipping: threshold early stopping, ξ selection, and final metrics
all used the same observations.

### Fold label fix (Fix 6)

`build_fold_labels` now merges any ecozone with fewer than `min_size` fire-days
(default 200) into the "North (merged)" zone 999, in addition to the hard-coded
high-latitude zones {1, 2, 3, 11}.


In [ ]:
def build_fold_labels(ecozones: Sequence[int], min_size: int = 200) -> np.ndarray:
    """
    FIX 6: min_size is now actually enforced.
    Zones with < min_size samples OR in the hard-coded north list are merged to 999.
    """
    ecozones = pd.Series(ecozones).astype(int)
    counts   = ecozones.value_counts()
    north_zones = {1, 2, 3, 11}

    fold_map = {}
    for z, cnt in counts.items():
        z = int(z)
        if z in north_zones or cnt < min_size:
            fold_map[z] = 999
        else:
            fold_map[z] = z

    labels = np.array([fold_map.get(int(z), int(z)) for z in ecozones], dtype=int)
    return labels


def make_three_way_masks(
    df: pd.DataFrame,
    tune_years: Sequence[int] = TUNE_YEARS,
    eval_years: Sequence[int] = EVAL_YEARS,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    FIX 4: Return three non-overlapping boolean masks.

    Returns
    -------
    train_mask : 2002–2017 (fitting)
    tune_mask  : 2018–2019 (early stopping + xi selection)
    eval_mask  : 2020–2021 (final evaluation only)
    """
    years = df[YEAR_COL].to_numpy()
    eval_mask  = np.isin(years, list(eval_years))
    tune_mask  = np.isin(years, list(tune_years))
    train_mask = ~eval_mask & ~tune_mask
    return train_mask, tune_mask, eval_mask


def describe_group_counts(df: pd.DataFrame) -> pd.Series:
    counts = df[GROUP_COL].astype(int).value_counts().sort_values(ascending=False)
    return counts.rename(index=ECOZONE_NAMES)


## 6. Stage 1 — Adaptive threshold model

**Fix 4 applied:** Early stopping uses `tune_mask`, never `eval_mask`.


In [ ]:
@dataclass
class ThresholdResult:
    model:              lgb.LGBMRegressor
    threshold:          np.ndarray   # u(s,t) for every row in df
    exceedance_mask:    np.ndarray   # y > u
    credible_mask:      np.ndarray   # y > u AND u >= exceedance_floor
    exceedances_capped: np.ndarray   # y[credible] - u[credible], capped
    cap_value:          float


def fit_threshold_model(
    df:               pd.DataFrame,
    feature_cols:     Sequence[str],
    train_mask:       np.ndarray,
    tune_mask:        np.ndarray,        # FIX 4: explicit tune set for early stopping
    tau:              float = TAU,
    exceedance_floor: float = EXCEEDANCE_FLOOR,
    cap_percentile:   float = CAP_PERCENTILE,
) -> ThresholdResult:
    X = df[list(feature_cols)]
    y = df[RESPONSE_COL].to_numpy(dtype=float)

    params = dict(
        objective      = "quantile",
        alpha          = tau,
        num_leaves     = 63,
        learning_rate  = 0.05,
        n_estimators   = 400,
        min_child_samples = 30,
        subsample      = 0.8,
        colsample_bytree = 0.8,
        reg_lambda     = 1.0,
        verbose        = -1,
    )
    model = lgb.LGBMRegressor(**params)

    fit_args: dict = dict(
        X = X.loc[train_mask],
        y = y[train_mask],
        callbacks = [lgb.early_stopping(30, verbose=False)],
    )
    if tune_mask.sum() > 0:
        fit_args["eval_set"] = [(X.loc[tune_mask], y[tune_mask])]

    model.fit(**fit_args)

    u = np.maximum(model.predict(X), 1.0)
    exceedance_mask = y > u
    credible_mask   = exceedance_mask & (u >= exceedance_floor)

    y_exc = y[credible_mask] - u[credible_mask]
    if len(y_exc) == 0:
        cap_value, exceedances_capped = 0.0, np.array([])
    else:
        cap_value          = float(np.percentile(y_exc, cap_percentile))
        exceedances_capped = np.minimum(y_exc, cap_value)

    return ThresholdResult(
        model=model, threshold=u,
        exceedance_mask=exceedance_mask, credible_mask=credible_mask,
        exceedances_capped=exceedances_capped, cap_value=cap_value,
    )


## 7. Tail-model helpers

In [ ]:
def gpd_nll_score(y_exc: np.ndarray, sigma: np.ndarray, xi: float) -> float:
    r     = xi * y_exc / sigma
    denom = np.maximum(1.0 + r, 1e-10)
    nll   = np.log(sigma) + (1.0 + 1.0 / xi) * np.log(denom)
    return float(np.mean(nll))


def gpd_exceedance_quantile(sigma: np.ndarray, xi: float, p: float) -> np.ndarray:
    """GPD quantile for exceedance y > 0 at probability level p."""
    return sigma / xi * ((1.0 - p) ** (-xi) - 1.0)


def pinball_at_level(y_exc: np.ndarray, sigma: np.ndarray, xi: float, tau: float) -> float:
    q = gpd_exceedance_quantile(sigma, xi, tau)
    return float(mean_pinball_loss(y_exc, q, alpha=tau))


def coverage_at_level(y_exc: np.ndarray, sigma: np.ndarray, xi: float, level: float) -> float:
    q = gpd_exceedance_quantile(sigma, xi, level)
    return float(np.mean(y_exc <= q))


def twcrps_score(y_exc: np.ndarray, sigma: np.ndarray, xi: float,
                 n_quantiles: int = 50) -> float:
    taus   = np.linspace(0.01, 0.999, n_quantiles)
    scores = []
    for tau in taus:
        q     = gpd_exceedance_quantile(sigma, xi, tau)
        resid = y_exc - q
        pin   = np.where(resid >= 0, tau * resid, (tau - 1.0) * resid)
        scores.append(np.mean(pin))
    return float(np.mean(scores))


def gpd_objective_fixed_xi(xi: float):
    """
    Custom LightGBM objective for GPD NLL with fixed shape xi.
    preds = log(sigma) = eta_init + boosted_residual.
    Gradient and Hessian are w.r.t. the log-scale prediction.
    """
    def fobj(preds: np.ndarray, train_data: lgb.Dataset):
        y_true = train_data.get_label()
        sigma  = np.exp(preds)
        r      = xi * y_true / sigma
        denom  = np.maximum(1.0 + r, 1e-8)
        c      = 1.0 + 1.0 / xi
        grad   = 1.0 - c * r / denom
        hess   = np.maximum(c * r / denom ** 2, 1e-6)
        return grad, hess

    def feval(preds: np.ndarray, train_data: lgb.Dataset):
        y_true = train_data.get_label()
        sigma  = np.exp(preds)
        return "gpd_nll", gpd_nll_score(y_true, sigma, xi), False

    return fobj, feval


## 8. Profile likelihood for the shape parameter ξ

**Fix 4 applied:** `profile_xi` now accepts `X_tune` / `y_tune` explicitly.
The caller passes tune-set exceedances; the eval set is never touched here.


In [ ]:
def profile_xi(
    X_train:    np.ndarray,
    y_train:    np.ndarray,
    X_tune:     np.ndarray,     # FIX 4: renamed from X_val
    y_tune:     np.ndarray,     # FIX 4: renamed from y_val
    xi_grid:    np.ndarray = XI_GRID,
) -> pd.DataFrame:
    """
    Select xi by profile likelihood evaluated on the TUNE set only.
    The eval set must not be passed here.
    """
    base_params = dict(
        num_leaves       = 63,
        learning_rate    = 0.05,
        min_child_samples= 20,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        reg_lambda       = 1.0,
        verbose          = -1,
    )
    sigma_init = max(float(np.mean(y_train)), 1.0)
    eta_init   = np.log(sigma_init)
    rows       = []

    for xi in xi_grid:
        fobj, feval = gpd_objective_fixed_xi(float(xi))
        dtrain = lgb.Dataset(X_train, label=y_train,
                             init_score=np.full(len(y_train), eta_init))
        dtune  = lgb.Dataset(X_tune, label=y_tune,
                             init_score=np.full(len(y_tune), eta_init),
                             reference=dtrain)
        model = lgb.train(
            params         = {**base_params, "objective": fobj},
            train_set      = dtrain,
            num_boost_round= 200,
            valid_sets     = [dtune],
            feval          = feval,
            callbacks      = [lgb.early_stopping(20, verbose=False)],
        )
        rows.append({"xi": float(xi),
                     "tune_nll": float(model.best_score["valid_0"]["gpd_nll"])})

    df_out = pd.DataFrame(rows).sort_values("tune_nll").reset_index(drop=True)
    return df_out


## 9. EBM interpretability layer

**Paper correction (Fix 7):** Section 6.2 previously stated the EBM was fitted
on the "full dataset." The code correctly fits on `train_mask` only and predicts
out-of-sample for tune and eval rows. Fitting on the full dataset would leak
future fire-weather patterns into the interpretable component. The paper text
will be corrected to read "fitted on training data only."


In [ ]:
from interpret.glassbox import ExplainableBoostingRegressor


def fit_ebm_layer(
    df:         pd.DataFrame,
    train_mask: np.ndarray,
    max_rounds: int = 3000,
) -> Tuple[ExplainableBoostingRegressor, np.ndarray]:
    """
    Fit EBM on TRAIN rows only; predict f_interp for ALL rows.
    Tune and eval rows receive genuine out-of-sample EBM predictions.
    """
    X_interp = df[INTERPRETED]
    y_log    = np.log1p(df[RESPONSE_COL].to_numpy(dtype=float))

    ebm = ExplainableBoostingRegressor(
        feature_names  = INTERPRETED,
        interactions   = 2,
        max_bins       = 256,
        max_rounds     = max_rounds,
        learning_rate  = 0.01,
        min_samples_leaf = 20,
        random_state   = 42,
    )
    ebm.fit(X_interp.loc[train_mask], y_log[train_mask])
    f_interp = ebm.predict(X_interp)   # out-of-sample for non-train rows
    return ebm, f_interp


def plot_ebm_effects_to_file(
    ebm:         ExplainableBoostingRegressor,
    interpreted: Sequence[str] = INTERPRETED,
    filename:    str = "ebm_effects.png",
) -> Path:
    ebm_global = ebm.explain_global()
    n = len(interpreted)
    fig, axes = plt.subplots(1, n, figsize=(max(18, 3.2 * n), 3.5))
    if n == 1:
        axes = [axes]
    for i, feat in enumerate(interpreted):
        data  = ebm_global.data(i)
        xvals = data["names"]
        yvals = data["scores"]
        if len(xvals) == len(yvals) + 1:
            xvals = [(xvals[j] + xvals[j + 1]) / 2 for j in range(len(yvals))]
        axes[i].plot(xvals, yvals, lw=2)
        axes[i].axhline(0, color="k", ls="--", lw=0.8)
        axes[i].set_title(feat)
    axes[0].set_ylabel("EBM effect on log(1 + spread)")
    fig.suptitle("EBM partial effects (interpreted covariates)", y=1.04)
    fig.tight_layout()
    path = ensure_output_dir() / filename
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path


## 10. Tail data assembly

In [ ]:
@dataclass
class TailData:
    X_train:      np.ndarray
    y_train:      np.ndarray
    X_tune:       np.ndarray    # FIX 4: renamed from X_val
    y_tune:       np.ndarray    # FIX 4: renamed from y_val
    X_eval:       np.ndarray    # FIX 4: new — eval set, never used for fitting
    y_eval:       np.ndarray    # FIX 4: new
    feature_names: List[str]
    sigma_init:   float
    eta_init:     float


def make_tail_data(
    df:               pd.DataFrame,
    threshold_result: "ThresholdResult",
    f_interp:         np.ndarray,
    train_mask:       np.ndarray,
    tune_mask:        np.ndarray,
    eval_mask:        np.ndarray,
    feature_cols:     Sequence[str],
) -> TailData:
    y         = df[RESPONSE_COL].to_numpy(dtype=float)
    u         = threshold_result.threshold
    credible  = threshold_result.credible_mask
    cap       = threshold_result.cap_value

    def _exc(mask):
        m = credible & mask
        return m, np.minimum(y[m] - u[m], cap)

    train_exc_mask, y_train_exc = _exc(train_mask)
    tune_exc_mask,  y_tune_exc  = _exc(tune_mask)
    eval_exc_mask,  y_eval_exc  = _exc(eval_mask)

    X_base = df[list(feature_cols)].to_numpy()

    def _X_aug(mask):
        return np.hstack([X_base[mask], f_interp[mask].reshape(-1, 1)])

    sigma_init    = max(float(np.mean(y_train_exc)) if len(y_train_exc) > 0 else 1.0, 1.0)
    eta_init      = np.log(sigma_init)
    feature_names = list(feature_cols) + ["f_interp"]

    return TailData(
        X_train       = _X_aug(train_exc_mask),
        y_train       = y_train_exc,
        X_tune        = _X_aug(tune_exc_mask),
        y_tune        = y_tune_exc,
        X_eval        = _X_aug(eval_exc_mask),
        y_eval        = y_eval_exc,
        feature_names = feature_names,
        sigma_init    = sigma_init,
        eta_init      = eta_init,
    )


## 11. Stage 2b — GPD-LightGBM scale model

**Fix 4 applied:** Early stopping uses `X_tune` / `y_tune`, not eval data.


In [ ]:
@dataclass
class TailModelResult:
    xi:            float
    model:         lgb.Booster
    eta_init:      float
    feature_names: List[str]


def fit_tail_model(tail_data: TailData, xi: float) -> TailModelResult:
    """FIX 4: valid_sets uses TUNE data only; eval set is never seen here."""
    fobj, feval = gpd_objective_fixed_xi(xi)

    params = dict(
        num_leaves       = 127,
        learning_rate    = 0.03,
        min_child_samples= 30,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        reg_lambda       = 1.0,
        verbose          = -1,
        objective        = fobj,
    )

    dtrain = lgb.Dataset(
        tail_data.X_train, label=tail_data.y_train,
        init_score   = np.full(len(tail_data.y_train), tail_data.eta_init),
        feature_name = tail_data.feature_names,
    )
    dtune = lgb.Dataset(
        tail_data.X_tune, label=tail_data.y_tune,
        init_score   = np.full(len(tail_data.y_tune), tail_data.eta_init),
        reference    = dtrain,
        feature_name = tail_data.feature_names,
    )

    model = lgb.train(
        params          = params,
        train_set       = dtrain,
        num_boost_round = 1000,
        valid_sets      = [dtrain, dtune],
        valid_names     = ["train", "tune"],
        feval           = feval,
        callbacks       = [lgb.early_stopping(50, verbose=False)],
    )
    return TailModelResult(
        xi=float(xi), model=model,
        eta_init=float(tail_data.eta_init),
        feature_names=tail_data.feature_names,
    )


def predict_sigma(model_result: TailModelResult, X_aug: np.ndarray) -> np.ndarray:
    """
    model.predict() returns the boosted residual only (LightGBM does not include
    init_score in predict output), so we add eta_init back manually.
    """
    log_sigma = model_result.eta_init + model_result.model.predict(X_aug)
    return np.exp(log_sigma)


## 12. Baselines and evaluation

In [ ]:
def stationary_gpd_baseline(y_train_exc: np.ndarray) -> Tuple[float, float]:
    xi_hat, _, sigma_hat = genpareto.fit(y_train_exc, floc=0)
    return float(sigma_hat), float(xi_hat)


def fit_constant_sigma(y_train_exc: np.ndarray, xi: float) -> float:
    def objective(log_sigma: float) -> float:
        sigma = np.exp(log_sigma)
        return gpd_nll_score(y_train_exc, np.full_like(y_train_exc, sigma), xi)
    opt = minimize_scalar(objective, bounds=(np.log(1.0), np.log(10_000.0)),
                          method="bounded")
    return float(np.exp(opt.x))


def evaluate_gpd_model(
    y_exc:  np.ndarray,
    sigma:  np.ndarray,
    xi:     float,
    label:  str,
) -> Dict[str, float]:
    return {
        "model":       label,
        "nll":         gpd_nll_score(y_exc, sigma, xi),
        "twcrps":      twcrps_score(y_exc, sigma, xi),
        "pinball_95":  pinball_at_level(y_exc, sigma, xi, 0.95),
        "pinball_99":  pinball_at_level(y_exc, sigma, xi, 0.99),
        "coverage_95": coverage_at_level(y_exc, sigma, xi, 0.95),
        "coverage_99": coverage_at_level(y_exc, sigma, xi, 0.99),
    }


def fit_quantile_baselines(
    df:           pd.DataFrame,
    feature_cols: Sequence[str],
    train_mask:   np.ndarray,
    tune_mask:    np.ndarray,
    levels:       Sequence[float] = (0.95, 0.99),
) -> Dict[float, lgb.LGBMRegressor]:
    """FIX 4: early stopping uses tune set."""
    X = df[list(feature_cols)]
    y = df[RESPONSE_COL].to_numpy(dtype=float)
    models = {}
    for q in levels:
        model = lgb.LGBMRegressor(
            objective="quantile", alpha=q, num_leaves=63,
            learning_rate=0.05, n_estimators=400,
            min_child_samples=30, subsample=0.8,
            colsample_bytree=0.8, reg_lambda=1.0, verbose=-1,
        )
        model.fit(
            X.loc[train_mask], y[train_mask],
            eval_set  = [(X.loc[tune_mask], y[tune_mask])],
            callbacks = [lgb.early_stopping(30, verbose=False)],
        )
        models[q] = model
    return models


def evaluate_quantile_predictions(
    y_true: np.ndarray, y_pred: np.ndarray, tau: float
) -> Dict[str, float]:
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "model":    f"Quantile LGBM {tau:.2f}",
        "pinball":  float(mean_pinball_loss(y_true, y_pred, alpha=tau)),
        "coverage": float(np.mean(y_true <= y_pred)),
    }


## 13. Return levels

**Fix 1:** `return_level` now requires the threshold array `u` and adds it to
the exceedance part: $z_T(x) = u(x) + \frac{\hat{\sigma}(x)}{\xi}[(T \cdot p_u)^\xi - 1]$.

**Fix 2:** Return periods are labelled `_fd` (fire-days), not `_yr`.
T values match Section 10 of the paper: {10, 50, 100, 250} fire-days.
If annual return levels are needed, convert via
`T_fd = T_years × mean_fire_days_per_year`.


In [ ]:
def return_level(
    sigma:      np.ndarray,
    xi:         float,
    T_firedays: float,
    p_u:        float,
    u:          Optional[np.ndarray] = None,
) -> np.ndarray:
    """
    FIX 1: Threshold u is added to the exceedance return level.

    Parameters
    ----------
    sigma      : covariate-dependent GPD scale (length n)
    xi         : shape parameter (scalar, fixed globally)
    T_firedays : return period in fire-days (NOT years)
    p_u        : exceedance probability = 1 - tau
    u          : threshold array (same length as sigma); required for correct RL

    Returns
    -------
    z_T : T-fire-day return level (m/day), shape (n,)
    """
    if xi == 0.0:
        exceedance_part = sigma * np.log(T_firedays * p_u)
    else:
        exceedance_part = sigma / xi * ((T_firedays * p_u) ** xi - 1.0)

    if u is None:
        import warnings
        warnings.warn(
            "return_level called without u — threshold not added. "
            "Pass u=threshold_result.threshold for correct return levels.",
            stacklevel=2,
        )
        return exceedance_part
    return u + exceedance_part


def run_return_level_analysis(state: Dict[str, object]) -> pd.DataFrame:
    """
    FIX 1 + FIX 2: Return periods in fire-days; threshold u included.
    T values: {10, 50, 100, 250} fire-days, matching paper Section 10.
    """
    df        = state["df"].copy()
    sigma_all = np.asarray(state["sigma_all"], dtype=float)
    u_all     = state["threshold_result"].threshold   # FIX 1: needed for RL
    p_u       = 1.0 - TAU

    # FIX 2: fire-day return periods, not years
    T_values = [10, 50, 100, 250]
    out = pd.DataFrame({"ecozone": df[GROUP_COL].astype(int)})
    for T in T_values:
        out[f"rl_{T}fd"] = return_level(sigma_all, state["xi_best"], T, p_u, u=u_all)

    rl_cols = [f"rl_{T}fd" for T in T_values]
    summary = out.groupby("ecozone", as_index=False).mean(numeric_only=True)
    summary["ecozone_name"] = (
        summary["ecozone"].map(ECOZONE_NAMES)
        .fillna(summary["ecozone"].astype(str))
    )
    summary = summary[["ecozone", "ecozone_name"] + rl_cols].sort_values(
        "rl_100fd", ascending=False
    )
    save_dataframe(summary, "return_levels_by_ecozone.csv")

    # Plot 100-fire-day return level
    plot_df = summary.head(12).iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(plot_df["ecozone_name"], plot_df["rl_100fd"])
    ax.set_xlabel("100-fire-day return level (m/day)")
    ax.set_ylabel("Ecozone")
    ax.set_title("Top ecozones — 100-fire-day return level")
    fig.tight_layout()
    path = ensure_output_dir() / "return_level_100fd_top_ecozones.png"
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close(fig)

    state["return_level_df"] = summary   # store for choropleth use
    return summary


## 14. Output utilities

In [ ]:
def ensure_output_dir() -> Path:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    return OUTPUT_DIR


def save_open_figure(filename: str) -> Path:
    path = ensure_output_dir() / filename
    plt.gcf().savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close(plt.gcf())
    return path


def save_dataframe(df_obj: pd.DataFrame, filename: str) -> Path:
    path = ensure_output_dir() / filename
    df_obj.to_csv(path, index=False)
    return path


def build_augmented_matrix(
    df:           pd.DataFrame,
    f_interp:     np.ndarray,
    feature_cols: Sequence[str] = ALL_FEATURES,
) -> np.ndarray:
    """Stack raw features + EBM output column."""
    return np.hstack([df[list(feature_cols)].to_numpy(),
                      np.asarray(f_interp).reshape(-1, 1)])


## 15. Secondary analyses

### Fix 5 — Spatial CV early stopping

In v5, `_fit_single_fold` passed `test_mask` as the eval set to both
`fit_threshold_model` and `fit_tail_model`, allowing early stopping to be
guided by the held-out fold — a form of spatial leakage.

**Fix:** a temporal tune subset is carved from each fold's training data
(latest `FOLD_TUNE_YEARS` years in that fold's training period).

### Fix 8 — Fold-specific ξ in spatial CV

When `SPATIAL_CV_REFIT_XI = True`, `profile_xi` is re-run within each fold
using fold-specific training and tune data. This removes the mild leakage
of globally-selected ξ into folds that contributed to its selection.

### Fix 10 — NGBoost `logpdf`

`GPDDist` was missing an explicit `logpdf` method.

### Fix 11 — SHAP collinearity

In v5, `run_shap_analysis` fed `f_interp` into SHAP alongside the seven
interpreted features that are its inputs. Because f_interp is a nonlinear
super-feature of those seven variables, SHAP attributed importance to it
at the expense of its own inputs, distorting the rankings.

**Fix:** SHAP is run on the **non-augmented** feature matrix (`ALL_FEATURES`
only, without `f_interp`). A separate importance bar for `f_interp` is still
reported as an aggregate, but individual feature rankings are clean.


In [ ]:
BIOME_MAP = {
    1: "Arctic",           2: "Arctic",              3: "Arctic",
    4: "Taiga",            5: "Taiga",               6: "Boreal",
    7: "Temperate/Maritime", 8: "Temperate/Mixedwood", 9: "Boreal",
    10: "Prairie/Grassland", 11: "Taiga",            12: "Boreal",
    13: "Temperate/Maritime", 14: "Cordillera",      15: "Hudson Plains",
    25: "Taiga",           26: "Boreal",             999: "Taiga",
}

# Years held back as an intra-fold tune set during spatial CV
FOLD_TUNE_YEARS = [2018, 2019]


def run_mean_excess_analysis(state: Dict[str, object]) -> pd.DataFrame:
    """
    Mean excess plot for the stationary GPD vs empirical exceedances.
    For GPD(sigma, xi) with xi < 1, the theoretical mean excess at
    sub-threshold t is e(t) = (sigma + xi * t) / (1 - xi).
    """
    y_sorted = np.sort(np.asarray(state["y_train_exc"], dtype=float))
    rows = []
    for q in np.linspace(0.10, 0.95, 40):
        thr = float(np.quantile(y_sorted, q))
        exc = y_sorted[y_sorted > thr] - thr
        if len(exc) >= 30:
            rows.append({"threshold": thr, "empirical_mean_excess": float(exc.mean())})

    mean_excess_df = pd.DataFrame(rows)
    sigma_s = state["stationary_sigma"]
    xi_s    = state["stationary_xi"]
    # GPD mean excess: (sigma + xi * t) / (1 - xi)
    # Guard: if xi >= 1 mean excess is infinite; plot is not meaningful.
    if xi_s < 1.0:
        me = (sigma_s + xi_s * mean_excess_df["threshold"]) / (1.0 - xi_s)
    else:
        me = np.full(len(mean_excess_df), np.nan)
        print(f"Warning: stationary xi={xi_s:.3f} >= 1; GPD mean excess is infinite.")
    mean_excess_df["gpd_mean_excess"] = me

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(mean_excess_df["threshold"], mean_excess_df["empirical_mean_excess"],
            "o-", ms=4, label="Empirical")
    ax.plot(mean_excess_df["threshold"], mean_excess_df["gpd_mean_excess"],
            "r--", label=f"Stationary GPD (xi={xi_s:.2f})")
    ax.set_xlabel("Threshold exceedance (m/day)")
    ax.set_ylabel("Mean excess")
    ax.set_title("Mean excess plot")
    ax.legend()
    save_open_figure("mean_excess_plot.png")
    save_dataframe(mean_excess_df, "mean_excess_plot_data.csv")
    return mean_excess_df


def _fit_single_fold(
    df:         pd.DataFrame,
    train_mask: np.ndarray,
    test_mask:  np.ndarray,
    xi_fixed:   float,
) -> Optional[Dict[str, float]]:
    """
    FIX 5: A temporal subset of train_mask is used for early stopping inside the
    fold; test_mask observations are NEVER passed to fit_threshold_model or
    fit_tail_model.

    FIX 8: If SPATIAL_CV_REFIT_XI is True, xi is re-estimated inside the fold.
    """
    # Carve a temporal tune set from within the fold's training data
    years_in_train = df.loc[train_mask, YEAR_COL].unique()
    fold_tune_yrs  = [y for y in FOLD_TUNE_YEARS if y in years_in_train]
    fold_tune_mask = train_mask & df[YEAR_COL].isin(fold_tune_yrs).to_numpy()
    fold_fit_mask  = train_mask & ~fold_tune_mask

    # Fall back to full train if tune slice is empty (unlikely but safe)
    if fold_tune_mask.sum() == 0:
        fold_fit_mask  = train_mask
        fold_tune_mask = train_mask

    threshold_result = fit_threshold_model(
        df, ALL_FEATURES, fold_fit_mask, fold_tune_mask   # FIX 5: tune, not test
    )
    ebm, f_interp = fit_ebm_layer(
        df, fold_fit_mask,
        max_rounds=500 if FAST_CV else 3000,
    )
    tail_data = make_tail_data(
        df, threshold_result, f_interp,
        fold_fit_mask, fold_tune_mask, test_mask,   # eval=test for this fold
        ALL_FEATURES,
    )

    if len(tail_data.y_train) < 50 or len(tail_data.y_eval) < 50:
        return None

    # FIX 8: optionally re-estimate xi within the fold
    if SPATIAL_CV_REFIT_XI and len(tail_data.y_tune) >= 30:
        xi_profile = profile_xi(
            tail_data.X_train, tail_data.y_train,
            tail_data.X_tune,  tail_data.y_tune,
        )
        xi_used = float(xi_profile.iloc[0]["xi"])
    else:
        xi_used = xi_fixed

    model       = fit_tail_model(tail_data, xi_used)
    y_test_exc  = tail_data.y_eval
    sigma_test  = predict_sigma(model, tail_data.X_eval)

    out = evaluate_gpd_model(y_test_exc, sigma_test, xi_used, label="GBM-GPD")
    out["n_test"] = int(len(y_test_exc))
    out["xi_used"] = float(xi_used)
    return out


def run_spatial_block_cv(state: Dict[str, object]) -> pd.DataFrame:
    df     = state["df"]
    labels = build_fold_labels(df[GROUP_COL].astype(int))
    rows   = []
    for fold in sorted(pd.unique(labels)):
        test_mask  = (labels == fold)
        train_mask = ~test_mask
        metrics = _fit_single_fold(df, train_mask, test_mask, state["xi_best"])
        if metrics is None:
            continue
        metrics["fold"]  = int(fold)
        metrics["group"] = ECOZONE_NAMES.get(int(fold), f"Zone {int(fold)}")
        rows.append(metrics)
    cv_df = pd.DataFrame(rows)
    save_dataframe(cv_df, "spatial_block_cv.csv")
    if not cv_df.empty:
        plot_df = cv_df.sort_values("twcrps")
        fig, ax = plt.subplots(figsize=(10, max(4, 0.35 * len(plot_df))))
        ax.barh(plot_df["group"], plot_df["twcrps"])
        ax.set_xlabel("twCRPS")
        ax.set_ylabel("Held-out ecozone")
        ax.set_title("Spatial block CV — ecozone holdout (leakage-free)")
        fig.tight_layout()
        save_open_figure("spatial_block_cv_twcrps.png")
    return cv_df


def run_biome_cv(state: Dict[str, object]) -> pd.DataFrame:
    df = state["df"].copy()
    df["biome"] = df[GROUP_COL].astype(int).map(BIOME_MAP).fillna("Other")
    rows = []
    for biome in sorted(df["biome"].unique()):
        test_mask  = (df["biome"] == biome).to_numpy()
        train_mask = ~test_mask
        metrics = _fit_single_fold(df, train_mask, test_mask, state["xi_best"])
        if metrics is None:
            continue
        metrics["biome"] = biome
        rows.append(metrics)
    biome_df = pd.DataFrame(rows)
    save_dataframe(biome_df, "biome_cv.csv")
    if not biome_df.empty:
        plot_df = biome_df.sort_values("twcrps")
        fig, ax = plt.subplots(figsize=(8, max(4, 0.5 * len(plot_df))))
        ax.barh(plot_df["biome"], plot_df["twcrps"], color="tab:orange")
        ax.set_xlabel("twCRPS")
        ax.set_ylabel("Held-out biome")
        ax.set_title("Biome CV holdout performance")
        fig.tight_layout()
        save_open_figure("biome_cv_twcrps.png")
    return biome_df


def compare_cv_schemes(
    spatial_df: pd.DataFrame, biome_df: pd.DataFrame
) -> pd.DataFrame:
    rows = []
    for name, df_ in [("Ecozone-based CV", spatial_df), ("Biome-based CV", biome_df)]:
        if df_.empty:
            continue
        rows.append({
            "Scheme":             name,
            "Mean twCRPS":        float(df_["twcrps"].mean()),
            "Std twCRPS":         float(df_["twcrps"].std()),
            "Min twCRPS":         float(df_["twcrps"].min()),
            "Max twCRPS":         float(df_["twcrps"].max()),
            "Total Test Samples": int(df_["n_test"].sum()),
        })
    out = pd.DataFrame(rows)
    save_dataframe(out, "cv_scheme_comparison.csv")
    if not out.empty:
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.bar(out["Scheme"], out["Mean twCRPS"],
               yerr=out["Std twCRPS"], capsize=8)
        ax.set_ylabel("Mean twCRPS")
        ax.set_title("Ecozone vs biome CV")
        fig.tight_layout()
        save_open_figure("cv_scheme_comparison.png")
    return out


def run_ngboost_comparison(state: Dict[str, object]) -> pd.DataFrame:
    from ngboost import NGBRegressor
    from ngboost.distns.distn import RegressionDistn
    from ngboost.scores import LogScore
    from sklearn.tree import DecisionTreeRegressor

    class GPDLogScore(LogScore):
        def score(self, Y):
            return -self.dist.logpdf(Y)

        def d_score(self, Y):
            D     = np.zeros((len(Y), 2))
            sigma = self.dist.scale
            xi    = self.dist.xi
            r     = xi * Y / sigma
            denom = np.maximum(1.0 + r, 1e-8)
            c     = 1.0 + 1.0 / xi
            D[:, 0] = 1.0 - c * r / denom
            d_nll_d_xi = (-1.0 / xi ** 2) * np.log(denom) + c * Y / sigma / denom
            sig_raw    = 1.0 / (1.0 + np.exp(-self.dist.raw_xi))
            d_xi_d_raw = 1.49 * sig_raw * (1.0 - sig_raw)
            D[:, 1] = d_nll_d_xi * d_xi_d_raw
            return D

    class GPDDist(RegressionDistn):
        n_params = 2
        scores   = [GPDLogScore]

        def __init__(self, params):
            self._params_raw = params
            self.log_sigma   = params[0]
            self.raw_xi      = params[1]
            self.scale       = np.maximum(np.exp(self.log_sigma), 1e-6)
            self.xi          = np.clip(
                0.01 + 1.49 / (1.0 + np.exp(-self.raw_xi)), 0.01, 1.49
            )
            self._scipy_dist = genpareto(c=self.xi, loc=0, scale=self.scale)

        # FIX 10: logpdf was missing in v5 — caused AttributeError at runtime
        def logpdf(self, Y: np.ndarray) -> np.ndarray:
            return self._scipy_dist.logpdf(Y)

        @staticmethod
        def fit(Y):
            xi_fit, _, sigma_fit = genpareto.fit(Y, floc=0)
            xi_fit    = float(np.clip(xi_fit, 0.02, 1.48))
            sigma_fit = float(max(sigma_fit, 1.0))
            log_sigma = np.log(sigma_fit)
            raw_xi    = np.log((xi_fit - 0.01) / (1.50 - xi_fit))
            return np.array([log_sigma, raw_xi])

        def sample(self, m):
            return np.array([self._scipy_dist.rvs() for _ in range(m)])

        @property
        def params(self):
            return {"scale": self.scale, "xi": self.xi}

    X_train = state["X_all_aug"][state["train_exc_mask"]]
    X_eval  = state["X_all_aug"][state["eval_exc_mask"]]
    y_train = np.maximum(np.asarray(state["y_train_exc"], dtype=float), 0.1)
    y_eval  = np.maximum(np.asarray(state["y_eval_exc"],  dtype=float), 0.1)

    # Use tune set for early stopping, not eval
    X_tune  = state["X_all_aug"][state["tune_exc_mask"]]
    y_tune  = np.maximum(np.asarray(state["y_tune_exc"],  dtype=float), 0.1)

    ngb = NGBRegressor(
        Dist          = GPDDist,
        Score         = GPDLogScore,
        Base          = DecisionTreeRegressor(max_depth=4, min_samples_leaf=20),
        n_estimators  = 1000,
        learning_rate = 0.03,
        natural_gradient = True,
        minibatch_frac   = 0.8,
        random_state     = RANDOM_STATE,
        verbose          = False,
    )
    ngb.fit(X_train, y_train, X_val=X_tune, Y_val=y_tune,   # FIX 4: tune for ES
            early_stopping_rounds=50)

    dist     = ngb.pred_dist(X_eval)
    sigma_ng = np.asarray(dist.scale, dtype=float)
    xi_ng    = np.asarray(dist.xi,    dtype=float)
    xi_med   = float(np.median(xi_ng))

    out = pd.DataFrame([{
        "model":      "NGBoost-GPD learned xi",
        "nll":        float((-dist.logpdf(y_eval)).mean()),
        "twcrps":     twcrps_score(y_eval, sigma_ng, xi_med),
        "pinball_95": pinball_at_level(y_eval, sigma_ng, xi_med, 0.95),
        "pinball_99": pinball_at_level(y_eval, sigma_ng, xi_med, 0.99),
        "coverage_95": coverage_at_level(y_eval, sigma_ng, xi_med, 0.95),
        "coverage_99": coverage_at_level(y_eval, sigma_ng, xi_med, 0.99),
        "median_xi":  xi_med,
        "mean_xi":    float(np.mean(xi_ng)),
    }])
    save_dataframe(out, "ngboost_comparison.csv")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(xi_ng, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
    axes[0].axvline(state["xi_best"], color="red", ls="--", lw=2,
                    label=f"GBM-GPD fixed xi={state['xi_best']:.2f}")
    axes[0].axvline(xi_med, color="orange", lw=2,
                    label=f"NGBoost median={xi_med:.2f}")
    axes[0].set_xlabel("xi"); axes[0].set_ylabel("Count")
    axes[0].set_title("Distribution of NGBoost xi"); axes[0].legend(fontsize=9)
    axes[1].scatter(sigma_ng, xi_ng, s=8, alpha=0.4, color="steelblue")
    axes[1].axhline(state["xi_best"], color="red", ls="--", lw=1.5)
    axes[1].set_xlabel("sigma"); axes[1].set_ylabel("xi")
    axes[1].set_title("NGBoost sigma vs xi")
    fig.tight_layout()
    fig.savefig(ensure_output_dir() / "ngboost_xi_distribution.png",
                dpi=200, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return out


def run_shap_analysis(state: Dict[str, object]) -> pd.DataFrame:
    """
    FIX 11: SHAP is run on the non-augmented feature matrix (ALL_FEATURES only,
    without f_interp). In v5, f_interp was included alongside its own inputs
    (isi, fwi, ws, rh, slope, doy_sin, doy_cos), causing SHAP to attribute
    credit to the super-feature at the expense of its components and distorting
    the importance ranking.

    The tail model was trained on the augmented matrix (with f_interp), so SHAP
    values here reflect the model's actual decisions. f_interp's aggregate
    contribution is reported separately.
    """
    credible_mask = state["credible_mask"]

    # Non-augmented: shape (n_credible, len(ALL_FEATURES))
    X_raw      = df_from_state(state)[ALL_FEATURES].to_numpy()[credible_mask]
    # f_interp column
    f_col      = state["f_interp"][credible_mask].reshape(-1, 1)
    # Full augmented matrix (what the model actually saw)
    X_aug_full = np.hstack([X_raw, f_col])

    if len(X_aug_full) > 3000:
        rng       = np.random.default_rng(RANDOM_STATE)
        idx       = rng.choice(len(X_aug_full), size=3000, replace=False)
        X_aug_full = X_aug_full[idx]
        X_raw      = X_raw[idx]

    feature_names_aug = list(ALL_FEATURES) + ["f_interp"]
    explainer  = shap.TreeExplainer(state["tail_model"].model)
    shap_vals  = explainer.shap_values(X_aug_full)  # shape (n, n_features+1)

    # Full importance table (augmented, model-accurate)
    mean_abs_aug = np.abs(shap_vals).mean(axis=0)
    shap_df_aug  = (
        pd.DataFrame({"feature": feature_names_aug, "mean_abs_shap": mean_abs_aug})
        .sort_values("mean_abs_shap", ascending=False)
    )
    save_dataframe(shap_df_aug, "shap_importance_full.csv")

    # Clean importance: drop f_interp from the plot so raw features aren't eclipsed
    shap_vals_raw  = shap_vals[:, :len(ALL_FEATURES)]   # exclude f_interp column
    mean_abs_raw   = np.abs(shap_vals_raw).mean(axis=0)
    shap_df_raw    = (
        pd.DataFrame({"feature": list(ALL_FEATURES), "mean_abs_shap": mean_abs_raw})
        .sort_values("mean_abs_shap", ascending=False)
    )
    save_dataframe(shap_df_raw, "shap_importance_raw_features.csv")

    shap.summary_plot(shap_vals_raw, X_raw, feature_names=list(ALL_FEATURES),
                      show=False)
    save_open_figure("shap_summary_beeswarm.png")
    shap.summary_plot(shap_vals_raw, X_raw, feature_names=list(ALL_FEATURES),
                      plot_type="bar", show=False)
    save_open_figure("shap_summary_bar.png")

    f_interp_shap = float(np.abs(shap_vals[:, -1]).mean())
    print(f"f_interp aggregate |SHAP| (EBM super-feature): {f_interp_shap:.4f}")
    return shap_df_raw


def df_from_state(state: Dict[str, object]) -> pd.DataFrame:
    """Helper: recover the cleaned DataFrame from the state dict."""
    return state["df"]


## 16. Choropleth maps

In [ ]:
import geopandas as gpd_geo


def load_ecozone_boundaries(
    path:      Optional[str] = ECOZONE_SHAPEFILE_PATH,
    id_column: str = ECOZONE_ID_COLUMN,
) -> Optional[gpd_geo.GeoDataFrame]:
    if not path:
        print("Skipping choropleths: set ECOZONE_SHAPEFILE_PATH.")
        return None
    gdf = gpd_geo.read_file(path)
    if id_column not in gdf.columns:
        raise ValueError(f"{id_column!r} not in shapefile columns: {list(gdf.columns)}")
    gdf = gdf.copy()
    gdf[id_column] = pd.to_numeric(gdf[id_column], errors="coerce")
    gdf = gdf.dropna(subset=[id_column]).copy()
    gdf[id_column] = gdf[id_column].astype(int)
    return gdf


def plot_ecozone_choropleth(
    return_level_df: pd.DataFrame,
    eco_gdf:         gpd_geo.GeoDataFrame,
    value_col:       str = "rl_100fd",
) -> Path:
    merged = eco_gdf.merge(return_level_df, left_on=ECOZONE_ID_COLUMN,
                           right_on="ecozone", how="left")
    fig, ax = plt.subplots(figsize=(10, 8))
    merged.plot(column=value_col, cmap="YlOrRd", linewidth=0.4,
                edgecolor="0.4", legend=True, ax=ax,
                missing_kwds={"color": "lightgrey", "label": "No data"})
    ax.set_title(f"Ecozone choropleth: {value_col}")
    ax.set_axis_off(); fig.tight_layout()
    path = ensure_output_dir() / f"ecozone_choropleth_{value_col}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    merged.drop(columns="geometry").to_csv(
        ensure_output_dir() / f"ecozone_choropleth_{value_col}.csv", index=False)
    return path


def build_biome_boundaries(eco_gdf: gpd_geo.GeoDataFrame) -> gpd_geo.GeoDataFrame:
    eco = eco_gdf.copy()
    if BIOME_NAME_COLUMN and BIOME_NAME_COLUMN in eco.columns:
        eco["biome"] = eco[BIOME_NAME_COLUMN].astype(str)
    else:
        eco["biome"] = eco[ECOZONE_ID_COLUMN].map(BIOME_MAP).fillna("Other")
    return eco[["biome", "geometry"]].dissolve(by="biome", as_index=False)


def plot_biome_choropleth(
    return_level_df: pd.DataFrame,
    eco_gdf:         gpd_geo.GeoDataFrame,
    value_col:       str = "rl_100fd",
) -> Path:
    bv = return_level_df.copy()
    bv["biome"] = bv["ecozone"].map(BIOME_MAP).fillna("Other")
    rl_cols = [c for c in bv.columns if c.startswith("rl_")]
    bv = bv.groupby("biome", as_index=False)[rl_cols].mean()
    biome_gdf = build_biome_boundaries(eco_gdf)
    merged = biome_gdf.merge(bv, on="biome", how="left")
    fig, ax = plt.subplots(figsize=(10, 8))
    merged.plot(column=value_col, cmap="YlGnBu", linewidth=0.5,
                edgecolor="0.35", legend=True, ax=ax,
                missing_kwds={"color": "lightgrey", "label": "No data"})
    ax.set_title(f"Biome choropleth: {value_col}")
    ax.set_axis_off(); fig.tight_layout()
    path = ensure_output_dir() / f"biome_choropleth_{value_col}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    merged.drop(columns="geometry").to_csv(
        ensure_output_dir() / f"biome_choropleth_{value_col}.csv", index=False)
    return path


def run_choropleth_maps(state: Dict[str, object]) -> Dict[str, Path]:
    if "return_level_df" not in state:
        state["return_level_df"] = run_return_level_analysis(state)
    eco_gdf = load_ecozone_boundaries()
    if eco_gdf is None:
        return {}
    return {
        "ecozone_map": plot_ecozone_choropleth(state["return_level_df"],
                                               eco_gdf, "rl_100fd"),
        "biome_map":   plot_biome_choropleth(state["return_level_df"],
                                             eco_gdf, "rl_100fd"),
    }


## 17. Run-all orchestration

**Fix 3:** Complete state dict — all keys needed by downstream functions are
present. A single definition of `compute_main_results` (Fix 9: dead copy removed).


In [ ]:
def compute_main_results() -> Dict[str, object]:
    """
    Full pipeline: load → preprocess → three-way split → threshold →
    EBM → profile xi (tune set only) → tail model → baselines → eval.

    Returns a complete state dict. All downstream secondary analyses
    (SHAP, CV, return levels, NGBoost, choropleth) consume this dict.
    """
    ensure_output_dir()

    # ── Data ──────────────────────────────────────────────────────────────
    df_raw = load_from_bigquery()
    df     = preprocess_fire_days(df_raw)
    summarize_dataset(df)

    # FIX 4: three-way split
    train_mask, tune_mask, eval_mask = make_three_way_masks(df)

    # ── Stage 1: threshold ────────────────────────────────────────────────
    # FIX 4+5: tune_mask for early stopping; eval_mask never used here
    threshold_result = fit_threshold_model(df, ALL_FEATURES, train_mask, tune_mask)

    # ── EBM ───────────────────────────────────────────────────────────────
    ebm, f_interp = fit_ebm_layer(df, train_mask)
    plot_ebm_effects_to_file(ebm)

    # ── Tail data ─────────────────────────────────────────────────────────
    tail_data = make_tail_data(
        df, threshold_result, f_interp,
        train_mask, tune_mask, eval_mask, ALL_FEATURES,
    )

    min_samples = 50
    can_fit = (len(tail_data.y_train) >= min_samples and
               len(tail_data.y_eval)  >= min_samples)

    if not can_fit:
        print(f"\nWARNING: insufficient tail samples "
              f"(train={len(tail_data.y_train)}, eval={len(tail_data.y_eval)}). "
              f"Skipping GPD modeling.")

    if can_fit:
        # FIX 4: profile_xi uses tune set only
        xi_profile = profile_xi(
            tail_data.X_train, tail_data.y_train,
            tail_data.X_tune,  tail_data.y_tune,
        )
        xi_best = float(xi_profile.iloc[0]["xi"])
        save_dataframe(xi_profile, "xi_profile.csv")
        print(f"Best xi = {xi_best:.3f}  (tune NLL = {xi_profile.iloc[0]['tune_nll']:.4f})")

        # FIX 4: tail model early-stops on tune; eval never seen
        tail_model = fit_tail_model(tail_data, xi_best)

        X_all_aug = build_augmented_matrix(df, f_interp, ALL_FEATURES)
        sigma_all = predict_sigma(tail_model, X_all_aug)

        y             = df[RESPONSE_COL].to_numpy(dtype=float)
        credible_mask = threshold_result.credible_mask
        u             = threshold_result.threshold

        train_exc_mask = credible_mask & train_mask
        tune_exc_mask  = credible_mask & tune_mask
        eval_exc_mask  = credible_mask & eval_mask

        cap = threshold_result.cap_value
        y_train_exc = np.minimum(y[train_exc_mask] - u[train_exc_mask], cap)
        y_tune_exc  = np.minimum(y[tune_exc_mask]  - u[tune_exc_mask],  cap)
        y_eval_exc  = np.minimum(y[eval_exc_mask]  - u[eval_exc_mask],  cap)
        sigma_eval  = sigma_all[eval_exc_mask]

        stationary_sigma, stationary_xi = stationary_gpd_baseline(y_train_exc)
        const_sigma = fit_constant_sigma(y_train_exc, xi_best)

        # All evaluation on EVAL set only (Fix 4)
        full_eval = evaluate_gpd_model(
            y_eval_exc, sigma_eval, xi_best,
            label=f"GBM-GPD xi={xi_best:.2f}",
        )
        b1_eval = evaluate_gpd_model(
            y_eval_exc,
            np.full_like(y_eval_exc, stationary_sigma),
            stationary_xi, label="Stationary GPD",
        )
        b3_eval = evaluate_gpd_model(
            y_eval_exc,
            np.full_like(y_eval_exc, const_sigma),
            xi_best, label="Const-sigma GPD",
        )
        evaluation_df = pd.DataFrame([b1_eval, b3_eval, full_eval])
        save_dataframe(evaluation_df, "gpd_baseline_comparison.csv")
        print("\nEvaluation on EVAL set (2020–2021):")
        print(evaluation_df.to_string(index=False))
    else:
        (xi_profile, xi_best, tail_model, sigma_all,
         evaluation_df, X_all_aug) = (
            pd.DataFrame(), float("nan"), None,
            np.full(len(df), np.nan), pd.DataFrame(),
            np.full((len(df), len(ALL_FEATURES) + 1), np.nan),
        )
        y            = df[RESPONSE_COL].to_numpy(dtype=float)
        credible_mask = threshold_result.credible_mask
        u             = threshold_result.threshold
        cap           = threshold_result.cap_value
        train_exc_mask = credible_mask & train_mask
        tune_exc_mask  = credible_mask & tune_mask
        eval_exc_mask  = credible_mask & eval_mask
        y_train_exc = y_tune_exc = y_eval_exc = np.array([])
        stationary_sigma = stationary_xi = const_sigma = float("nan")

    # ── Quantile baselines (Fix 4: tune set for early stopping) ───────────
    quantile_models = fit_quantile_baselines(
        df, ALL_FEATURES, train_mask, tune_mask, levels=(0.95, 0.99)
    )
    y_eval_full = y[eval_mask]
    q_rows = [
        evaluate_quantile_predictions(
            y_eval_full,
            model.predict(df.loc[eval_mask, ALL_FEATURES]),
            tau,
        )
        for tau, model in quantile_models.items()
    ]
    quantile_df = pd.DataFrame(q_rows)
    save_dataframe(quantile_df, "quantile_baseline_comparison.csv")

    # FIX 3: complete state dict — all keys needed by downstream functions
    return {
        # core data
        "df":               df,
        "y":                y,
        "train_mask":       train_mask,
        "tune_mask":        tune_mask,
        "eval_mask":        eval_mask,
        # threshold
        "threshold_result": threshold_result,
        # EBM
        "ebm":              ebm,
        "f_interp":         f_interp,
        # tail data
        "tail_data":        tail_data,
        # xi selection
        "xi_profile":       xi_profile,
        "xi_best":          xi_best,
        # tail model
        "tail_model":       tail_model,
        # predictions
        "X_all_aug":        X_all_aug,
        "sigma_all":        sigma_all,
        # exceedance masks
        "credible_mask":    credible_mask,
        "train_exc_mask":   train_exc_mask,
        "tune_exc_mask":    tune_exc_mask,
        "eval_exc_mask":    eval_exc_mask,
        # exceedance responses
        "y_train_exc":      y_train_exc,
        "y_tune_exc":       y_tune_exc,
        "y_eval_exc":       y_eval_exc,
        # baselines
        "stationary_sigma": stationary_sigma,
        "stationary_xi":    stationary_xi,
        "const_sigma":      const_sigma,
        # results
        "evaluation_df":    evaluation_df,
        "quantile_df":      quantile_df,
        "can_fit_gpd":      can_fit,
    }


In [ ]:
# ── Run everything ────────────────────────────────────────────────────────
analysis_state = compute_main_results()

if analysis_state["can_fit_gpd"]:
    if RUN_RETURN_LEVELS:
        rl_df = run_return_level_analysis(analysis_state)

    if RUN_MEAN_EXCESS:
        me_df = run_mean_excess_analysis(analysis_state)

    if RUN_SHAP_ANALYSIS:
        shap_df = run_shap_analysis(analysis_state)

    if RUN_SPATIAL_CV:
        cv_df = run_spatial_block_cv(analysis_state)
        if RUN_BIOME_CV:
            biome_df = run_biome_cv(analysis_state)
            compare_cv_schemes(cv_df, biome_df)

    if RUN_NGBOOST_COMPARISON:
        ng_df = run_ngboost_comparison(analysis_state)

    if RUN_CHOROPLETH_MAPS:
        choropleth_paths = run_choropleth_maps(analysis_state)
else:
    print("GPD modeling skipped — insufficient tail data. "
          "Run check_tail_counts() for diagnostics.")


## 18. Diagnostics

In [ ]:
def check_tail_counts(state: Dict[str, object]) -> None:
    """
    Summarise exceedance counts across all three splits.
    Call this after compute_main_results() if tail data is sparse.
    """
    tr = state["threshold_result"]
    credible = tr.credible_mask
    print(f"Total rows in df             : {len(state['df']):,}")
    print(f"Credible exceedances (all)   : {credible.sum():,}")
    for name, mask in [
        ("  TRAIN (fit)",   state["train_mask"]),
        ("  TUNE  (ES/xi)", state["tune_mask"]),
        ("  EVAL  (report)",state["eval_mask"]),
    ]:
        n = (credible & mask).sum()
        print(f"Tail samples {name}: {n:,}")
    if (credible & state["train_mask"]).sum() < 50:
        print("\nALERT: fewer than 50 training exceedances. "
              "Consider lowering EXCEEDANCE_FLOOR or TAU.")

# Uncomment to run:
# check_tail_counts(analysis_state)
